# PV SCADA + Meteorological Dataset Builder

proyek ini menggunakan dataset opensource yang disediakan pada : `DOI:10.17632/9fdvyjrf67.2`

Menyusun dataset terintegrasi dengan kolom:
`timestamp, site_id, irradiance_w_m2, ambient_temp_c, power_kw, voltage_v, current_a`.

Mapping variabel yang digunakan:
- `ghi_wm2` → `irradiance_w_m2`
- `ambient_temp` → `ambient_temp_c`
- `dc_power_kw` → `power_kw`
- `dc_voltage_v` → `voltage_v`
- `dc_current_a` → `current_a`

Timestamp SCADA dan meteorological data diseragamkan terlebih dahulu, kemudian digabung berdasarkan timestamp.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("PV SCADA data")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCADA_FILE = DATA_DIR / "Inverter SCADA.csv"
METEO_FILE = DATA_DIR / "Meteorological data.csv"

OUTPUT_FILE = OUTPUT_DIR / "pv_monitoring_dataset.csv"


In [2]:
scada = pd.read_csv(SCADA_FILE)
meteo = pd.read_csv(METEO_FILE)

print("SCADA shape:", scada.shape)
print("Meteorological shape:", meteo.shape)

print("\nSCADA columns:")
print(scada.columns.tolist())

print("\nMeteorological columns:")
print(meteo.columns.tolist())


SCADA shape: (34560, 9)
Meteorological shape: (25920, 13)

SCADA columns:
['timestamp', 'inverter_id', 'dc_power_kw', 'ac_power_kw', 'dc_voltage_v', 'dc_current_a', 'module_temp_c', 'inverter_efficiency', 'status_code']

Meteorological columns:
['station_id', 'reading_time', 'ghi_wm2', 'dni_wm2', 'dhi_wm2', 'ambient_temp', 'temp_unit', 'wind_speed_ms', 'relative_humidity_pct', 'barometric_pressure_hpa', 'solar_zenith_angle_deg', 'cloud_opacity_pct', 'precipitation_mm']


In [3]:
display(scada.head())
display(meteo.head())

print("SCADA dtypes:")
print(scada.dtypes)

print("\nMeteorological dtypes:")
print(meteo.dtypes)


,timestamp,inverter_id,dc_power_kw,ac_power_kw,dc_voltage_v,dc_current_a,module_temp_c,inverter_efficiency,status_code
0,2025-04-01 00:00:00,INV-01,0.0,0.0,0.0,0.0,21.81,NaN,NORMAL
1,2025-04-01 00:30:00,INV-01,0.0,0.0,0.0,0.0,23.11,NaN,NORMAL
2,2025-04-01 01:00:00,INV-01,0.0,0.0,0.0,0.0,22.41,NaN,NORMAL
3,2025-04-01 01:30:00,INV-01,0.0,0.0,0.0,0.0,19.84,NaN,NORMAL
4,2025-04-01 02:00:00,INV-01,0.0,0.0,0.0,0.0,21.14,NaN,NORMAL


,station_id,reading_time,ghi_wm2,dni_wm2,dhi_wm2,ambient_temp,temp_unit,wind_speed_ms,relative_humidity_pct,barometric_pressure_hpa,solar_zenith_angle_deg,cloud_opacity_pct,precipitation_mm
0,WS-North,2025-04-01 00:00:00,2.975,1.975,15.808,19.99,C,8.656,64.60,1026.34,75.077,38.34,0.107
1,WS-North,2025-04-01 00:15:00,0.254,2.284,3.017,15.26,C,6.542,51.35,1018.90,79.000,52.28,0.127
2,WS-North,2025-04-01 00:30:00,6.236,0.545,6.747,17.29,C,10.748,39.47,1011.39,43.376,19.23,0.266
3,WS-North,2025-04-01 00:45:00,6.142,0.678,1.746,15.87,C,11.092,70.66,1011.79,64.941,27.66,0.166
4,WS-North,2025-04-01 01:00:00,2.294,0.599,0.610,16.73,C,1.473,71.81,1020.33,63.319,27.04,0.601


SCADA dtypes:
timestamp               object
inverter_id             object
dc_power_kw            float64
ac_power_kw            float64
dc_voltage_v           float64
dc_current_a           float64
module_temp_c          float64
inverter_efficiency    float64
status_code             object
dtype: object

Meteorological dtypes:
station_id                  object
reading_time                object
ghi_wm2                    float64
dni_wm2                    float64
dhi_wm2                    float64
ambient_temp               float64
temp_unit                   object
wind_speed_ms              float64
relative_humidity_pct      float64
barometric_pressure_hpa    float64
solar_zenith_angle_deg     float64
cloud_opacity_pct          float64
precipitation_mm           float64
dtype: object


In [4]:
# Mapping kolom sumber -> nama kolom dataset akhir
SCADA_MAP = {
    "dc_power_kw": "power_kw",
    "dc_voltage_v": "voltage_v",
    "dc_current_a": "current_a",
}

METEO_MAP = {
    "ghi_wm2": "irradiance_w_m2",
    "ambient_temp": "ambient_temp_c",
}

SCADA_TIMESTAMP = "timestamp"
METEO_TIMESTAMP = "reading_time"

required_scada = [SCADA_TIMESTAMP, "inverter_id"] + list(SCADA_MAP.keys())
required_meteo = [METEO_TIMESTAMP] + list(METEO_MAP.keys())

missing_scada = [c for c in required_scada if c not in scada.columns]
missing_meteo = [c for c in required_meteo if c not in meteo.columns]

if missing_scada:
    raise KeyError(f"Kolom SCADA tidak ditemukan: {missing_scada}")

if missing_meteo:
    raise KeyError(f"Kolom meteorological tidak ditemukan: {missing_meteo}")


In [5]:
scada[SCADA_TIMESTAMP] = pd.to_datetime(scada[SCADA_TIMESTAMP], errors="coerce")
meteo[METEO_TIMESTAMP] = pd.to_datetime(meteo[METEO_TIMESTAMP], errors="coerce")

# Hapus timestamp yang gagal diparse
scada = scada.dropna(subset=[SCADA_TIMESTAMP]).copy()
meteo = meteo.dropna(subset=[METEO_TIMESTAMP]).copy()

# Urutkan sebelum proses time alignment
scada = scada.sort_values(["inverter_id", SCADA_TIMESTAMP])
meteo = meteo.sort_values(METEO_TIMESTAMP)

print("SCADA time range:", scada[SCADA_TIMESTAMP].min(), "->", scada[SCADA_TIMESTAMP].max())
print("Meteo time range:", meteo[METEO_TIMESTAMP].min(), "->", meteo[METEO_TIMESTAMP].max())


SCADA time range: 2025-04-01 00:00:00 -> 2025-06-29 23:30:00
Meteo time range: 2025-04-01 00:00:00 -> 2025-06-29 23:45:00


In [6]:
scada_selected = scada[
    [SCADA_TIMESTAMP, "inverter_id", *SCADA_MAP.keys()]
].rename(columns=SCADA_MAP)

meteo_selected = meteo[
    [METEO_TIMESTAMP, *METEO_MAP.keys()]
].rename(columns=METEO_MAP)

scada_selected = scada_selected.rename(columns={SCADA_TIMESTAMP: "timestamp"})
meteo_selected = meteo_selected.rename(columns={METEO_TIMESTAMP: "reading_time"})

print(scada_selected.head())
print(meteo_selected.head())

            timestamp inverter_id  power_kw  voltage_v  current_a
0 2025-04-01 00:00:00      INV-01       0.0        0.0        0.0
1 2025-04-01 00:30:00      INV-01       0.0        0.0        0.0
2 2025-04-01 01:00:00      INV-01       0.0        0.0        0.0
3 2025-04-01 01:30:00      INV-01       0.0        0.0        0.0
4 2025-04-01 02:00:00      INV-01       0.0        0.0        0.0
             reading_time  irradiance_w_m2  ambient_temp_c
0     2025-04-01 00:00:00            2.975           19.99
17280 2025-04-01 00:00:00            7.106           18.99
8640  2025-04-01 00:00:00            6.212           14.76
1     2025-04-01 00:15:00            0.254           15.26
17281 2025-04-01 00:15:00            3.702           18.04


In [7]:
scada_selected = scada_selected.sort_values("timestamp").reset_index(drop=True)
meteo_selected = meteo_selected.sort_values("reading_time").reset_index(drop=True)

dataset = pd.merge_asof(
    scada_selected,
    meteo_selected,
    left_on="timestamp",
    right_on="reading_time",
    direction="nearest",
    tolerance=pd.Timedelta(minutes=2, seconds=30),
)


final_columns = [
    "timestamp",
    "inverter_id",
    "irradiance_w_m2",
    "ambient_temp_c",
    "power_kw",
    "voltage_v",
    "current_a",
]

dataset = dataset[final_columns]

print("Final shape:", dataset.shape)
display(dataset.head(10))


Final shape: (34560, 7)


C:\Users\Rifqi\AppData\Local\Temp\ipykernel_1736\424255592.py:10: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  tolerance=pd.Timedelta(minutes=2, seconds=30),


,timestamp,inverter_id,irradiance_w_m2,ambient_temp_c,power_kw,voltage_v,current_a
0,2025-04-01 00:00:00,INV-01,6.212,14.76,0.0,0.0,0.0
1,2025-04-01 00:00:00,INV-07,6.212,14.76,0.0,0.0,0.0
2,2025-04-01 00:00:00,INV-06,6.212,14.76,0.0,0.0,0.0
3,2025-04-01 00:00:00,INV-05,6.212,14.76,0.0,0.0,0.0
4,2025-04-01 00:00:00,INV-04,6.212,14.76,0.0,0.0,0.0
5,2025-04-01 00:00:00,INV-03,6.212,14.76,0.0,0.0,0.0
6,2025-04-01 00:00:00,INV-02,6.212,14.76,0.0,0.0,0.0
7,2025-04-01 00:00:00,INV-08,6.212,14.76,0.0,0.0,0.0
8,2025-04-01 00:30:00,INV-07,9.885,16.50,0.0,0.0,0.0
9,2025-04-01 00:30:00,INV-05,9.885,16.50,0.0,0.0,0.0


In [8]:
print("Jumlah site:", dataset["inverter_id"].nunique())
print("Timestamp unik:", dataset["timestamp"].nunique())

print("\nMissing values:")
display(dataset.isna().sum().to_frame("missing"))

print("\nDuplicate rows:", dataset.duplicated().sum())
print("Duplicate site-timestamp:", dataset.duplicated(["inverter_id", "timestamp"]).sum())

print("\nBasic statistics:")
display(dataset[[
    "irradiance_w_m2",
    "ambient_temp_c",
    "power_kw",
    "voltage_v",
    "current_a",
]].describe().T)


Jumlah site: 8
Timestamp unik: 4320

Missing values:


,missing
timestamp,0
inverter_id,0
irradiance_w_m2,0
ambient_temp_c,0
power_kw,180
voltage_v,0
current_a,0



Duplicate rows: 0
Duplicate site-timestamp: 0

Basic statistics:


,count,mean,std,min,25%,50%,75%,max
irradiance_w_m2,34560.0,290.175041,335.960684,0.005,5.66575,97.8115,574.17325,1204.3690
ambient_temp_c,34560.0,36.251435,23.386199,9.220,21.04000,26.3600,34.77250,109.8800
power_kw,34380.0,706.640287,7456.111821,0.000,0.00000,11.3772,74.06210,143736.8685
voltage_v,34560.0,312.504720,299.864358,0.000,0.00000,581.5950,600.63250,620.0000
current_a,34560.0,0.062241,0.074107,0.000,0.00000,0.0215,0.12280,0.2983


In [9]:
dataset.to_csv(OUTPUT_FILE, index=False)

print(f"Dataset berhasil disimpan: {OUTPUT_FILE}")
print(f"Rows: {len(dataset):,}")
print(f"Columns: {dataset.columns.tolist()}")


Dataset berhasil disimpan: output\pv_monitoring_dataset.csv
Rows: 34,560
Columns: ['timestamp', 'inverter_id', 'irradiance_w_m2', 'ambient_temp_c', 'power_kw', 'voltage_v', 'current_a']


## Catatan metodologis

1. `dc_power_kw`, `dc_voltage_v`, dan `dc_current_a` dipertahankan sebagai variabel DC agar konsisten dengan sumber SCADA.
2. `ghi_wm2` digunakan sebagai representasi irradiance yang paling umum, sehingga output diberi nama `irradiance_w_m2`.
3. `ambient_temp` digunakan sebagai temperatur lingkungan, bukan temperatur modul.
4. Alignment menggunakan timestamp terdekat dengan toleransi 2,5 menit; data yang tidak memiliki pasangan dalam toleransi tersebut tidak diimputasi.
5. Jika SCADA memiliki lebih dari satu inverter untuk satu site, `site_id` dapat muncul beberapa kali pada timestamp yang sama. Dalam kasus tersebut, agregasi ke level site perlu ditentukan secara terpisah dan tidak boleh dilakukan otomatis sebelum memahami struktur dataset.